# Source E: Medi-Cal FFS provider list

Exploratory inspection only. Run the code cells from top to bottom with **Shift+Enter**. The source CSVs use Windows-1252 text encoding; the loader below specifies it so names with accented characters can be read.

## 1. Find the data and import pandas

In [2]:
from pathlib import Path
import pandas as pd

relative_partition_dir = Path(
    "data/provider-side/E-mediCal-fee-for-service-enrolled-providers/"
    "zipped_bc_too_big_for_github"
)

repo_root = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / relative_partition_dir).is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the Source E data from the notebook's current folder: "
        f"{Path.cwd()}. Start Jupyter with the repository folder as its root."
    )

partition_dir = repo_root / relative_partition_dir
print("Source E folder:", partition_dir)

Source E folder: C:\Users\Benjamin Tran\.copilot\repos\IEHP-CEO-Competition\data\provider-side\E-mediCal-fee-for-service-enrolled-providers\zipped_bc_too_big_for_github


## 2. Load the two CSV parts

Blank fields are treated as missing. Literal values such as `N/A` are left as provided in the source.

In [3]:
csv_files = [
    partition_dir / "medi-cal-ffs-provider-listing_10_26_26_PART 1 first 200k line.csv",
    partition_dir / "medi-cal-ffs-provider-listing_10_26_26_PART 2 next 180k line.csv",
]

providers_df = pd.concat(
    [
        pd.read_csv(
            path,
            encoding="cp1252",
            low_memory=False,
            keep_default_na=False,
            na_values=[""],
        )
        for path in csv_files
    ],
    ignore_index=True,
)

print("Loaded both CSV parts into providers_df.")

Loaded both CSV parts into providers_df.


## 3. Dataset overview: shape and columns

In [4]:
print("Shape (rows, columns):", providers_df.shape)
print("\nColumn names:")
print(providers_df.columns.tolist())

Shape (rows, columns): (380896, 47)

Column names:
['Provider_Number', 'NPI', 'NPI_Type', 'Owner_Number', 'Service_Location_Number', 'Legal_Name', 'Enroll_Status_Eff_DT', 'Provider_Taxonomy', 'NEMT_NMT_Provider_Type', 'ANC_Provider_Type', 'FI_Provider_Type_CD', 'FI_Provider_Type', 'Provider_License', 'FI_Provider_Specialty_CD', 'FI_Provider_Specialty', 'Out_of_State_Indicator', 'In_Out_State', 'Address_Attention', 'Address', 'Address2', 'City', 'State', 'ZIP', 'ZIP_4', 'DHCS_County_CD', 'FIPS_County_CD', 'County', 'Phone_Number', 'Medicaid_Patients', 'CHIP_Patients', 'Telehealth_Services', 'Provider_Website', 'Threshold_Languages', 'Other_Languages', 'Acc_Exam_Room', 'Acc_Exterior_Building', 'Acc_Interior_Building', 'Acc_Parking', 'Acc_Restroom', 'Acc_Medical_Equipment', 'Acc_Patient_Areas', 'Acc_Patient_Diagnostic', 'Provider_Source', 'Billing_Provider', 'Latitude', 'Longitude', 'CountyName']


## 4. Data types and first five rows

In [5]:
display(providers_df.dtypes.rename("dtype").to_frame())
display(providers_df.head())

,dtype
Provider_Number,object
NPI,float64
NPI_Type,str
Owner_Number,float64
Service_Location_Number,float64
Legal_Name,str
Enroll_Status_Eff_DT,str
Provider_Taxonomy,str
NEMT_NMT_Provider_Type,str
ANC_Provider_Type,str


,Provider_Number,NPI,NPI_Type,Owner_Number,Service_Location_Number,Legal_Name,Enroll_Status_Eff_DT,Provider_Taxonomy,NEMT_NMT_Provider_Type,ANC_Provider_Type,...,Acc_Parking,Acc_Restroom,Acc_Medical_Equipment,Acc_Patient_Areas,Acc_Patient_Diagnostic,Provider_Source,Billing_Provider,Latitude,Longitude,CountyName
0,1003000639.0,1.003001e+09,Type 1,1.0,1.0,"BENHARASH, PEYMAN MD",1/19/2011 0:00,No Taxonomy Provided,N/A,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,FFS Non-Billing Provider,34.063722,-118.443467,Los Angeles County
1,1003000563.0,1.003001e+09,Type 2,1.0,1.0,"CHUN, RICHARD B D MD INC",4/1/2002 0:00,207RG0100X,N/A,Gastroenterology,...,NaN,NaN,NaN,NaN,NaN,1.0,FFS Billing Provider,38.734427,-121.271995,Placer County
2,1003000563.0,1.003001e+09,Type 2,1.0,1.0,"CHUN, RICHARD B D MD INC",4/1/2002 0:00,207RG0100X,N/A,Gastroenterology,...,NaN,NaN,NaN,NaN,NaN,1.0,FFS Billing Provider,38.734427,-121.271995,Placer County
3,1003001157.0,1.003001e+09,Type 1,1.0,1.0,"KIM, DAVID C MD",7/18/2003 0:00,No Taxonomy Provided,N/A,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,FFS Non-Billing Provider,34.099394,-117.421262,San Bernardino County
4,1003000225.0,1.003000e+09,Type 1,1.0,1.0,"MORGAN, ANNA C A MD",10/15/2008 0:00,No Taxonomy Provided,N/A,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,FFS Non-Billing Provider,37.891776,-122.058050,Contra Costa County


## 5. Missing-value counts

These counts reflect blank CSV fields only; placeholder strings like `N/A` are not counted as missing.

In [6]:
display(
    providers_df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

,missing_count
Provider_Number,8
NPI,57
NPI_Type,57
Owner_Number,8
Service_Location_Number,8
Legal_Name,8
Enroll_Status_Eff_DT,8
Provider_Taxonomy,8
NEMT_NMT_Provider_Type,8
ANC_Provider_Type,123368


## 6. Candidate columns, unique values, and ZIP type

In [7]:
column_candidates = {
    "Provider ID or NPI": ["Provider_Number", "NPI", "NPI_Type"],
    "Provider name": ["Legal_Name"],
    "County": ["County", "CountyName", "DHCS_County_CD", "FIPS_County_CD"],
    "ZIP code": ["ZIP", "ZIP_4"],
    "City": ["City"],
    "Provider type": ["FI_Provider_Type_CD", "FI_Provider_Type", "ANC_Provider_Type"],
    "Specialty": ["FI_Provider_Specialty_CD", "FI_Provider_Specialty"],
}

print("Candidate columns present:")
for label, candidates in column_candidates.items():
    present = [column for column in candidates if column in providers_df.columns]
    print(f"{label}: {present}")

print("\nDataset as of: October 26, 2026 (from datapackage metadata; not a CSV column)")

for label, column in [
    ("County", "County"),
    ("Coordinate-derived county", "CountyName"),
    ("Provider type", "FI_Provider_Type"),
    ("Specialty", "FI_Provider_Specialty"),
]:
    if column in providers_df.columns:
        print(f"\nUnique {label} values ({column}):")
        print(providers_df[column].dropna().unique().tolist())

if "ZIP" in providers_df.columns:
    print("\nZIP pandas dtype:", providers_df["ZIP"].dtype)
    print("ZIP sample values:", providers_df["ZIP"].dropna().head(10).tolist())
    zip_is_text = pd.api.types.is_string_dtype(providers_df["ZIP"])
    print("ZIP is treated as:", "text" if zip_is_text else "non-text / likely numeric")

Candidate columns present:
Provider ID or NPI: ['Provider_Number', 'NPI', 'NPI_Type']
Provider name: ['Legal_Name']
County: ['County', 'CountyName', 'DHCS_County_CD', 'FIPS_County_CD']
ZIP code: ['ZIP', 'ZIP_4']
City: ['City']
Provider type: ['FI_Provider_Type_CD', 'FI_Provider_Type', 'ANC_Provider_Type']
Specialty: ['FI_Provider_Specialty_CD', 'FI_Provider_Specialty']

Dataset as of: October 26, 2026 (from datapackage metadata; not a CSV column)

Unique County values (County):
['Los Angeles', 'Placer', 'San Bernardino', 'Contra Costa', 'N/A', 'Sacramento', 'Imperial', 'Santa Clara', 'Monterey', 'Alameda', 'Orange', 'Ventura', 'Sonoma', 'UNKNOWN', 'Kern', 'Stanislaus', 'Merced', 'San Diego', 'San Mateo', 'Solano', 'Tuolumne', 'San Francisco', 'Fresno', 'Santa Cruz', 'Riverside', 'Nevada', 'El Dorado', 'Tulare', 'Kings', 'Santa Barbara', 'San Joaquin', 'Sutter', 'San Luis Obispo', 'Yolo', 'Marin', 'Shasta', 'Humboldt', 'San Benito', 'Napa', 'Butte', 'Mendocino', 'Mariposa', 'Del Norte',

## Field guide and limitations

Based on the accompanying `datapackage.json` schema:

- `Provider_Number` is the PMF provider identifier and can be either an NPI or a Medi-Cal legacy number. `NPI` is the separate CMS identifier; `NPI_Type` distinguishes individual (Type 1) from organization (Type 2). `Owner_Number` and `Service_Location_Number` identify ownership and service-location records.
- `Legal_Name` is the name reported in the Provider Master File. `Enroll_Status_Eff_DT` is the enrollment effective date, not the snapshot date.
- `FI_Provider_Type_CD` / `FI_Provider_Type` are PMF provider type code/description. `FI_Provider_Specialty_CD` / `FI_Provider_Specialty` are PMF specialty code/description. `Provider_Taxonomy` is PAVE-reported taxonomy; `ANC_Provider_Type` is a mapped network-certification type.
- `Address`, `Address2`, `City`, `State`, `ZIP`, and `ZIP_4` describe the service location. `DHCS_County_CD` and `FIPS_County_CD` are county codes. The schema gives no description for `County`; `CountyName` is explicitly based on latitude/longitude coordinates.
- `Latitude` and `Longitude` are coordinates. `Provider_License` is a reported state license number for specified provider categories; blanks mean license data is unavailable, and a dash may be shown for out-of-state or other providers.
- `Billing_Provider` distinguishes FFS billing from FFS non-billing providers. `Medicaid_Patients` is a reported new-patient acceptance field, with metadata meanings N/F/M/B. It is not inferred here and must not be treated as equivalent to enrollment. `CHIP_Patients` and `Telehealth_Services` are separate reported flags.
- `Out_of_State_Indicator` / `In_Out_State` represent location status. `NEMT_NMT_Provider_Type` describes approved non-emergency or non-medical transportation type. Language and `Acc_*` fields describe reported languages and accessibility features. `Provider_Source` is marked for DHCS internal use in the schema.

The metadata says this is a point-in-time Medi-Cal FFS enrollment dataset as of October 26, 2026. It excludes managed-care providers and dental FFS providers, may contain incomplete or inaccurate records, and may have a 1-2 day quality-assurance lag. The source schema does not define a UIS field; no UIS status should be inferred.

Before interpreting provider-per-member rates, confirm the intended row grain (provider, owner, or service location), the meaning/authority of `County` versus coordinate-derived `CountyName`, and how to handle legacy IDs and missing/literal placeholder values. ZIP codes are geographic identifiers and should remain text; the inspection cell reports the dtype actually loaded. The initial inspection section above does not clean, filter, deduplicate, visualize, or export the source data. The access-analysis section below filters derived working tables by the selected month and region without modifying the source files.

## 7. Access-to-care analysis setup

The competition guide recommends comparing one month of Medi-Cal members by ZIP (Source A) with enrolled FFS providers by service ZIP (Source E), using Source H to identify Riverside and San Bernardino ZIPs. September 2026 is used below as the guide recommends August or September; it is marked preliminary. The crosswalk is the latest copy included here (June 2026). ZIPs crossing county lines are flagged rather than allocated. These steps create analysis-specific tables without changing the source data.

### 7a. Build the Inland Empire ZIP list from Source H

In [ ]:
def normalize_zip(values):
    """Normalize ZIPs in derived analysis columns while retaining them as text."""
    zip_text = values.astype("string").str.strip()
    valid_zip = zip_text.str.fullmatch(r"\d{1,5}", na=False)
    return zip_text.where(valid_zip).str.zfill(5)

crosswalk_path = (
    repo_root
    / "data/provider-side/H-county-zip-crosswalk/ZIP-COUNTY_062026.xlsx"
)
county_zip_df = pd.read_excel(
    crosswalk_path,
    dtype={"zip": "string", "geoid": "string"},
)
county_zip_df["ZIP"] = normalize_zip(county_zip_df["zip"])
county_zip_df["County_GEOID"] = county_zip_df["geoid"].astype("string").str.zfill(5)

ie_counties = {
    "06065": "Riverside",
    "06071": "San Bernardino",
}
county_zip_df["County"] = county_zip_df["County_GEOID"].map(ie_counties)

all_counties_per_zip = (
    county_zip_df.dropna(subset=["ZIP", "County_GEOID"])
    .groupby("ZIP")["County_GEOID"]
    .nunique()
)
ie_zip_map = (
    county_zip_df.loc[
        county_zip_df["County_GEOID"].isin(ie_counties),
        ["ZIP", "County"],
    ]
    .dropna(subset=["ZIP"])
    .groupby("ZIP", as_index=False)
    .agg(IE_Counties=("County", lambda values: ", ".join(sorted(set(values)))))
)
ie_zip_map["Crosswalk_County_Count"] = (
    ie_zip_map["ZIP"].map(all_counties_per_zip).fillna(0).astype(int)
)
ie_zip_map["Crosses_County_Boundary"] = ie_zip_map["Crosswalk_County_Count"] > 1
ie_zips = set(ie_zip_map["ZIP"])

print("Inland Empire ZIPs in the June 2026 crosswalk:", len(ie_zips))
print(
    "IE ZIPs linked to more than one county:",
    int(ie_zip_map["Crosses_County_Boundary"].sum()),
)
display(ie_zip_map.sort_values("ZIP").head(15))

### 7b. Prepare September 2026 Medi-Cal member counts (Source A)

Source A includes a Total row as well as male and female rows, so this cell sums the two sex-specific rows and excludes Total to avoid double counting. Suppressed/blank counts remain missing rather than being treated as zero.

In [ ]:
members_path = (
    repo_root
    / "data/demand-side/A-medical-eligible-counts-by-ZIP/"
    "calendar-years-2020-to-the-most-recent-reportable-month.csv"
)
members_raw_df = pd.read_csv(
    members_path,
    encoding="cp1252",
    dtype={"Zip Code": "string", "Total Eligibles": "string"},
    keep_default_na=False,
)

selected_month = "2026-09"
month_rows = members_raw_df.loc[
    (members_raw_df["Month of Eligibility"] == selected_month)
    & (members_raw_df["Sex"].isin(["Female", "Male"]))
].copy()
month_rows["ZIP"] = normalize_zip(month_rows["Zip Code"])
month_rows["Eligible_Count"] = pd.to_numeric(
    month_rows["Total Eligibles"].str.replace(",", "", regex=False),
    errors="coerce",
)
month_rows = month_rows.loc[month_rows["ZIP"].isin(ie_zips)].copy()

sex_counts_by_zip = (
    month_rows.groupby(["ZIP", "Sex"])["Eligible_Count"]
    .sum(min_count=1)
    .unstack("Sex")
)
for sex in ["Female", "Male"]:
    if sex not in sex_counts_by_zip.columns:
        sex_counts_by_zip[sex] = pd.NA
sex_counts_by_zip["Medi-Cal Members"] = sex_counts_by_zip[["Female", "Male"]].sum(
    axis=1,
    min_count=2,
)

members_by_zip_ie = (
    ie_zip_map.set_index("ZIP")
    .join(sex_counts_by_zip[["Female", "Male", "Medi-Cal Members"]])
    .reset_index()
    .sort_values("ZIP")
)

print("Selected month:", selected_month)
print("Reported status:", sorted(month_rows["Status"].dropna().unique().tolist()))
print("Date of data:", sorted(month_rows["Date of Data"].dropna().unique().tolist()))
print("Rows for selected month and IE ZIPs:", len(month_rows))
print(
    "IE ZIP sex rows with an annotation code:",
    int(month_rows["Annotation Code"].str.strip().ne("").sum()),
)
print("ZIP rows with a missing sex-specific count:", int(sex_counts_by_zip[["Female", "Male"]].isna().any(axis=1).sum()))
display(members_by_zip_ie)

### 7c. Inspect Source E provider types and service locations

This inventory uses the provider-type and specialty descriptions reported by DHCS. It does not yet assign primary-care, specialty, or mental-health groupings; the challenge guide does not prescribe exact code lists. Multiple service-location records are audited but not removed.

In [ ]:
provider_columns = [
    "Provider_Number",
    "NPI",
    "NPI_Type",
    "Owner_Number",
    "Service_Location_Number",
    "FI_Provider_Type_CD",
    "FI_Provider_Type",
    "FI_Provider_Specialty_CD",
    "FI_Provider_Specialty",
    "ZIP",
    "City",
    "County",
    "Billing_Provider",
]
identifier_dtypes = {column: "string" for column in provider_columns}
provider_analysis_df = pd.concat(
    [
        pd.read_csv(
            path,
            usecols=provider_columns,
            dtype=identifier_dtypes,
            encoding="cp1252",
            keep_default_na=False,
            na_values=[""],
        )
        for path in csv_files
    ],
    ignore_index=True,
)
provider_analysis_df["ZIP"] = normalize_zip(provider_analysis_df["ZIP"])
providers_in_ie_zip_df = provider_analysis_df.loc[
    provider_analysis_df["ZIP"].isin(ie_zips)
].copy()

provider_type_inventory = (
    providers_in_ie_zip_df.groupby(
        ["FI_Provider_Type_CD", "FI_Provider_Type"],
        dropna=False,
    )
    .agg(
        source_rows=("Provider_Number", "size"),
        unique_provider_numbers=("Provider_Number", "nunique"),
        unique_npis=("NPI", "nunique"),
    )
    .reset_index()
    .sort_values("unique_provider_numbers", ascending=False)
)
specialty_inventory = (
    providers_in_ie_zip_df.groupby(
        ["FI_Provider_Specialty_CD", "FI_Provider_Specialty"],
        dropna=False,
    )
    .agg(
        source_rows=("Provider_Number", "size"),
        unique_provider_numbers=("Provider_Number", "nunique"),
    )
    .reset_index()
    .sort_values("unique_provider_numbers", ascending=False)
)

provider_type_by_zip = (
    providers_in_ie_zip_df.groupby(
        ["ZIP", "FI_Provider_Type_CD", "FI_Provider_Type"],
        dropna=False,
    )["Provider_Number"]
    .nunique()
    .rename("unique_provider_numbers")
    .reset_index()
)

provider_locations_by_id = (
    providers_in_ie_zip_df.dropna(subset=["Provider_Number"])
    .groupby("Provider_Number")
    .agg(
        service_locations=("Service_Location_Number", "nunique"),
        service_zips=("ZIP", "nunique"),
        source_rows=("Provider_Number", "size"),
    )
)
repeated_provider_ids = provider_locations_by_id.loc[
    (provider_locations_by_id["service_locations"] > 1)
    | (provider_locations_by_id["service_zips"] > 1)
]

print("Source E records in IE ZIPs:", len(providers_in_ie_zip_df))
print(
    "Unique provider numbers:",
    providers_in_ie_zip_df["Provider_Number"].nunique(),
)
print("Provider types reported in IE ZIPs:")
display(provider_type_inventory)
print("Specialties reported in IE ZIPs:")
display(specialty_inventory)
print("Provider-number counts by ZIP and reported type (first 30 rows):")
display(provider_type_by_zip.sort_values(["ZIP", "FI_Provider_Type"]).head(30))
print(
    "Provider numbers with multiple locations or ZIPs:",
    len(repeated_provider_ids),
)
display(
    repeated_provider_ids.sort_values(
        ["service_zips", "service_locations"],
        ascending=False,
    ).head(20)
)


### Interpretation before calculating rates

The member ZIP counts are preliminary and suppressed cells remain missing; neither should be silently interpreted as a complete count. Source E enrollment does not mean a provider is accepting new FFS patients, and the guide recommends treating it as a best-case supply estimate. The crosswalk flags ZIPs with more than one county relationship; no fractional allocation is applied here. Source E excludes dental providers, which the guide says to analyze separately with Source F.

Before calculating provider-per-1,000-member ratios, choose which reported provider types/specialties to include in primary care, key specialty, and mental-health groups, and confirm whether `Provider_Number` is the desired counting unit or whether provider NPIs / service locations should be counted differently. The inventory above exposes the raw types and repeated-location issue without deduplicating the source.